# Phase 3 - Kaggle Pipeline

OCR on Kaggle T4 -> vLLM on remote H100 -> Vietnamese summary

## Installation

In [1]:
import os
def install_if_missing(import_name, pip_cmd):
    try:
        __import__(import_name); print(f"'{import_name}' Existed")
    except ImportError:
        print(f"Installing: {pip_cmd}")
        from IPython import get_ipython;
        ip = get_ipython()
        if ip:
            ip.run_line_magic('pip', f'install {pip_cmd}')
        else: 
            os.system(f'pip install {pip_cmd}')

install_if_missing('cv2', 'opencv-python')

# paddleocr 3.7.0
install_if_missing('paddleocr', 'paddleocr==3.7.0')

# Remove stale CPU paddle build, install matching GPU build for CUDA 13.0
os.system('pip uninstall -y paddlepaddle')
os.system('pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/')

install_if_missing('vietocr', 'vietocr')
install_if_missing('openai', 'openai')

# Force a clean, ABI-consistent Pillow install LAST — vietocr's deps (albumentations)
# silently pull in a mismatched Pillow build that breaks JPEG encoding otherwise
os.system('pip uninstall -y pillow Pillow')
os.system('pip install --no-cache-dir "pillow==11.3.0"')

print("--- paddle packages ---")
os.system('pip list | grep -i paddle')
print("--- pillow version ---")
os.system('pip show pillow | grep Version')

'cv2' Existed
Installing: paddleocr==3.7.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 1.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 2.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 14.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 24.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 75.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu130/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 966.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 918.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 45.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 46.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 MB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 420.9/420.9 MB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 23.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
rmm-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
cuml-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
pylibraft-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
cuvs-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
cudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
torch 2.10.0+cu128 requires cuda-bindings==12.9.4; platform_system == "Linux", but you have cuda-bindings 13.0.3 which is incompatible.


Installing: vietocr
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.9/133.9 kB 1.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 7.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 33.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 26.6 MB/s eta 0:00:00
  Created wheel for gdown: filename=gdown-4.4.0-py3-none-any.whl size=14845 sha256=0f61ba4f1e310d667cdafa13e7766172e0cedb50d4a53de2c06dd20e99719171
  Stored in directory: /root/.cache/pip/wheels/cc/dd/9e/edc126b0a309c8ceb1a4f52f30aa1d95b5fad66c02f314fed5
  Created wheel for prefetch-generator: filename=prefetch_generator-1.0.1-py3-none-any.whl size=3987 sha256=83a3a12cee4127c723131d9a8ced213d18961676c1534edeae50cd39bd5ac427
  Stored in directory: /root/.c

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vietocr 0.3.13 requires pillow==10.2.0, but you have pillow 11.3.0 which is incompatible.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


--- paddle packages ---
paddleocr                                3.7.0
paddlepaddle-gpu                         3.3.0
paddlex                                  3.7.2
--- pillow version ---
Version: 11.3.0


0

In [2]:
from PIL import Image
import io
img = Image.new('RGB', (100, 100), color='red')
buf = io.BytesIO()
img.save(buf, format='JPEG', quality=85)
print("Pillow JPEG save OK, size:", len(buf.getvalue()))

Pillow JPEG save OK, size: 826


In [3]:
import paddle
print(paddle.device.is_compiled_with_cuda())
print(paddle.device.get_device())

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


True
gpu:0


## Bug Fixes

In [4]:
import os
import numpy as np
import PIL

# Critical: Disable PIR executor (fixes OneDNN PirAttribute errors)
os.environ["FLAGS_enable_pir_in_executor"] = "0"
os.environ["FLAGS_use_mkldnn"] = "0"

# Fix PIL._util (removed in newer Pillow)
if not hasattr(PIL, '_util'):
    class _util: pass
    PIL._util = _util
if not hasattr(PIL._util, 'is_directory'):
    PIL._util.is_directory = os.path.isdir

# Fix np.sctypes (removed in NumPy 2.x)
try:
    _ = np.sctypes
except AttributeError:
    np.sctypes = {
        "float": [np.float16, np.float32, np.float64],
        "int": [np.int8, np.int16, np.int32, np.int64],
        "uint": [np.uint8, np.uint16, np.uint32, np.uint64],
        "complex": [np.complex64, np.complex128]
    }
print("Bug fixes applied")


Bug fixes applied


## Imports

In [5]:
import json, re, base64
from io import BytesIO
from queue import Queue
from threading import Thread, Lock

import pandas as pd
import cv2
import torch
from paddleocr import TextDetection
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg
from tqdm.notebook import tqdm
from openai import OpenAI
HAS_GPU = torch.cuda.is_available()
print(f'GPU available: {HAS_GPU}')


/usr/local/lib/python3.12/dist-packages/gdown/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


GPU available: True


## Load Dataset

In [6]:
DATASET_PATH = '/kaggle/input/datasets/tonioz/imagedataset/images'
EXTENSIONS = ('.jpg', '.jpeg', '.png')
image_files = []
for dirname, _, files in os.walk(DATASET_PATH):
    for f in files:
        if f.lower().endswith(EXTENSIONS):
            image_files.append(os.path.join(dirname, f))

test_df = pd.DataFrame({
    'image_id': [os.path.splitext(os.path.basename(f))[0] for f in image_files],
    'image_path': image_files,
})
print(f'Images: {len(test_df)}')


Images: 1202


## Preprocessing

In [7]:
def classify_image(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mb = gray.mean()
    if mb > 200 or (gray > 240).sum()/gray.size > 0.30: 
        return 'overexposed'
    elif mb < 60 or (gray < 30).sum()/gray.size > 0.30: 
        return 'underexposed'
    elif cv2.Laplacian(gray, cv2.CV_64F).var() < 100: 
        return 'blurry'
    elif gray.std() < 42: 
        return 'low_contrast'
    elif cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)[:,:,1].std() > 75: 
        return 'complex'
    return 'normal'

def gamma_correct(img, g=1.3):
    table = np.array([(i/255.0)**(1.0/g)*255 for i in range(256)]).astype('uint8')
    return cv2.LUT(img, table)

def preprocess(img_pil, max_dim=1536, min_dim=800):
    bgr = np.array(img_pil.convert('RGB'))[:,:,::-1]
    h,w = bgr.shape[:2]
    if max(h,w) > max_dim: 
        scale = max_dim/max(h,w)
    elif max(h,w) < min_dim: 
        scale = min_dim/max(h,w)
    else: 
        scale = 1.0
    bgr = cv2.resize(bgr, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_CUBIC)
    cat = classify_image(bgr)
    k = np.array([[0,-0.5,0],[-0.5,3,-0.5],[0,-0.5,0]])

    def clahe(img, clip):
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        lab[:,:,0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8,8)).apply(lab[:,:,0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    
    if cat == 'normal': 
        bgr = cv2.filter2D(clahe(bgr,2.0), -1, k)
    elif cat == 'overexposed': 
        bgr = gamma_correct(clahe(bgr,3.5), 0.7)
    elif cat == 'underexposed': 
        bgr = gamma_correct(clahe(bgr,3.0), 1.5)
    elif cat == 'low_contrast':
        lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
        lab[:,:,0] = cv2.normalize(lab[:,:,0], None, 0, 255, cv2.NORM_MINMAX)
        lab[:,:,0] = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(lab[:,:,0])
        bgr = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        bgr = cv2.filter2D(bgr, -1, k)
    elif cat == 'blurry':
        g = cv2.GaussianBlur(bgr, (0,0), sigmaX=3)
        bgr = cv2.addWeighted(bgr, 1.5, g, -0.5, 0)
        bgr = clahe(bgr, 2.0)
    elif cat == 'complex':
        bgr = cv2.bilateralFilter(bgr, d=9, sigmaColor=75, sigmaSpace=75)
        bgr = cv2.filter2D(clahe(bgr,2.5), -1, k)
    return bgr[:,:,::-1]

def postprocess_ocr(text):
    if not text: return ''
    t = re.sub(r'\s+', ' ', text).strip().split()
    if not t: return ''
    r = [t[0]]
    for tok in t[1:]:
        if tok.lower() != r[-1].lower(): r.append(tok)
    return ' '.join(r)
print('Preprocessing loaded')


Preprocessing loaded


## OCR (PaddleOCR 3.7.0+ with PIR fix)

In [8]:
Detector = TextDetection(
    model_name="PP-OCRv5_server_det",
    device="gpu:0",
    limit_side_len=1536,
    limit_type="max",
    thresh=0.3,
    box_thresh=0.3,
    unclip_ratio=2.0,
)

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv5_server_det`.
[2026-07-31 05:54:06,539] [    INFO] _client.py:1025 - HTTP Request: GET https://huggingface.co/api/models/PaddlePaddle/PP-OCRv5_server_det/revision/main "HTTP/1.1 200 OK"


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

[2026-07-31 05:54:06,760] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/PaddlePaddle/PP-OCRv5_server_det/resolve/ca867c897ecbca8873081573a802ad70d499cb94/inference.json "HTTP/1.1 307 Temporary Redirect"
[2026-07-31 05:54:06,768] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/PaddlePaddle/PP-OCRv5_server_det/ca867c897ecbca8873081573a802ad70d499cb94/inference.json "HTTP/1.1 200 OK"
[2026-07-31 05:54:06,776] [    INFO] _client.py:1025 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/PaddlePaddle/PP-OCRv5_server_det/ca867c897ecbca8873081573a802ad70d499cb94/inference.json "HTTP/1.1 200 OK"
[2026-07-31 05:54:06,841] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggingface.co/PaddlePaddle/PP-OCRv5_server_det/resolve/ca867c897ecbca8873081573a802ad70d499cb94/.gitattributes "HTTP/1.1 307 Temporary Redirect"
[2026-07-31 05:54:06,848] [    INFO] _client.py:1025 - HTTP Request: HEAD https://huggi

In [9]:
config = Cfg.load_config_from_name('vgg_transformer')
config['device'] = 'cuda:0' if HAS_GPU else 'cpu'
config['predictor']['beamsearch'] = False
recognizer = Predictor(config)
print('VietOCR loaded')


Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth


100%|██████████| 548M/548M [00:02<00:00, 212MB/s] 
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
18533it [00:04, 3859.42it/s]


VietOCR loaded


In [10]:
import torch
import numpy as np
from vietocr.tool.translate import translate as _vt_translate, process_input as _vt_process

REC_MAX_SEQ = 96
REC_BATCH   = 24


def recognize_batch(pil_crops, max_seq_length=REC_MAX_SEQ):
    """
    Nhận diện nhiều crop trong một lượt GPU thật sự.
    Sort theo chiều rộng rồi pad về max của từng batch — thay vì bucket
    theo chiều rộng chính xác (khiến batch size thường = 1).
    Trả về (list[str], list[float]) đúng thứ tự đầu vào.
    """
    if not pil_crops:
        return [], []

    cfg  = recognizer.config
    h    = cfg['dataset']['image_height']
    wmin = cfg['dataset']['image_min_width']
    wmax = cfg['dataset']['image_max_width']
    dev  = cfg['device']

    tensors = [_vt_process(im, h, wmin, wmax) for im in pil_crops]
    order   = sorted(range(len(tensors)), key=lambda i: tensors[i].shape[-1])

    texts = [''] * len(pil_crops)
    probs = [0.0] * len(pil_crops)

    for k in range(0, len(order), REC_BATCH):
        part  = order[k:k + REC_BATCH]
        group = [tensors[i] for i in part]
        wb    = max(t.shape[-1] for t in group)
        # pad bằng 1.0 = trắng, vì process_input chuẩn hoá về [0,1]
        padded = [torch.nn.functional.pad(t, (0, wb - t.shape[-1]), value=1.0)
                  for t in group]
        batch = torch.cat(padded, 0).to(dev)

        sents, cprobs = _vt_translate(batch, recognizer.model,
                                      max_seq_length=max_seq_length)
        for j, i in enumerate(part):
            texts[i] = recognizer.vocab.decode(sents[j].tolist())
            probs[i] = float(cprobs[j])

    return texts, probs


print(f'recognize_batch ready (max_seq_length={REC_MAX_SEQ}, batch={REC_BATCH})')

recognize_batch ready (max_seq_length=96, batch=24)


In [11]:
def Crop_Padding(image, bbox, pad=8):
    box = np.array(bbox, dtype=int)
    x_min = max(0, np.min(box[:, 0]) - pad)
    x_max = min(image.shape[1], np.max(box[:, 0]) + pad)
    y_min = max(0, np.min(box[:, 1]) - pad)
    y_max = min(image.shape[0], np.max(box[:, 1]) + pad)
    return image[y_min:y_max, x_min:x_max]

def Sort_Boxes(boxes):
    if boxes is None or len(boxes) == 0: return []
    boxes = sorted(boxes, key=lambda b: b[0][1])
    threshold = np.median([abs(b[2][1] - b[0][1]) for b in boxes]) * 0.3
    sorted_boxes, cur_line, base_y = [], [boxes[0]], boxes[0][0][1]
    for box in boxes[1:]:
        if abs(box[0][1] - base_y) <= threshold: cur_line.append(box)
        else:
            sorted_boxes.extend(sorted(cur_line, key=lambda b: b[0][0]))
            cur_line, base_y = [box], box[0][1]
    if cur_line:
        sorted_boxes.extend(sorted(cur_line, key=lambda b: b[0][0]))
    return sorted_boxes

MAX_BOXES = 16
MIN_CROP  = 10


def poly_area(p):
    p = np.asarray(p, dtype=np.float64)
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


def run_ocr(img_cv2):
    r = Detector.predict(img_cv2)
    if r is None or len(r) == 0 or r[0] is None:
        return '', []
    res = r[0]
    boxes = res['dt_polys'] if 'dt_polys' in res else res.get('dt_boxes', [])
    if boxes is None or len(boxes) == 0:
        return '', []

    if MAX_BOXES and len(boxes) > MAX_BOXES:
        boxes = sorted(boxes, key=poly_area, reverse=True)[:MAX_BOXES]

    boxes = Sort_Boxes(boxes)
    if not len(boxes):
        return '', []

    # cắt trước, nhận diện sau — một lượt GPU cho cả ảnh
    crops, metas = [], []
    for b in boxes:
        crop = Crop_Padding(img_cv2, b)
        hc, wc = crop.shape[:2]
        if hc < MIN_CROP or wc < MIN_CROP:
            continue
        crops.append(Image.fromarray(crop)) 
        metas.append((wc * hc, b))

    if not crops:
        return '', []

    texts_raw, probs = recognize_batch(crops)

    texts, bdata = [], []
    for text, prob, (area, box) in zip(texts_raw, probs, metas):
        text = text.strip()
        if len(text) > 1 and not text.isdigit():
            texts.append(text)
            bdata.append({'text': text, 'area': area, 'box': box,
                          'prob': prob})
    return postprocess_ocr(' '.join(texts)), bdata
print('Success')


Success


## VLM Engine - Remote vLLM API

In [12]:
N_CTX = 16   # OCR lines passed to the model, ranked by box area


def build_ocr_context(ocr_text, box_data=None):
    if not ocr_text or not ocr_text.strip():
        return 'Không phát hiện chữ nào trong ảnh.'
    if not box_data:
        return 'Chữ phát hiện được: ' + ocr_text

    ranked = sorted(box_data, key=lambda b: b['area'], reverse=True)[:N_CTX]
    lines = [f'{i}. {b["text"]}' for i, b in enumerate(ranked, 1)]
    return 'Chữ phát hiện được (theo độ nổi bật):\n' + '\n'.join(lines)


PROMPT_SUMMARY = '''Bạn viết mô tả ngắn cho ảnh thu thập từ mạng xã hội Việt Nam.

## Nhiệm vụ
Viết một đoạn mô tả ngắn gọn bằng tiếng Việt CÓ DẤU về nội dung ảnh.

## Quy tắc
1. Nêu bối cảnh chính: ảnh sản phẩm, banner quảng cáo, ảnh livestream, ảnh review, ảnh chụp màn hình, ảnh phong cảnh, ảnh người...
2. TÊN NHÃN HÀNG và TÊN SẢN PHẨM: giữ nguyên chính tả và nguyên ký tự đúng như xuất hiện trong ảnh.
3. CÁC DÒNG CHỮ KHÁC: diễn đạt lại ý bằng lời của bạn. KHÔNG trích dẫn nguyên văn, KHÔNG dùng dấu ngoặc kép, KHÔNG chép lại từng dòng chữ trong ảnh.
4. Nếu một dòng OCR đọc ra vô nghĩa, sai chính tả nặng hoặc không thành câu: BỎ QUA hoàn toàn. Không nhắc đến, không đoán lại nội dung của nó.
5. Chỉ mô tả những gì CÓ trong ảnh. KHÔNG viết câu nói rằng ảnh không có hoặc thiếu thứ gì.
6. Viết khẳng định. KHÔNG dùng "có thể là", "có vẻ", "dường như", "không rõ", "không thể xác định". Không chắc thì bỏ hẳn chi tiết đó.
7. KHÔNG bịa nhãn hàng, sản phẩm, giá, con số hay chương trình khuyến mãi không nhìn thấy trong ảnh.
8. KHÔNG nhắc đến nhiệm vụ này và KHÔNG dùng các từ như FMCG, OCR, "hình ảnh này", "mô tả".
9. Nếu ảnh KHÔNG có chữ đọc được VÀ KHÔNG có nhãn hàng/sản phẩm nào: trả về chuỗi rỗng, không xuất bất kỳ ký tự nào.
10. Chỉ xuất đúng đoạn mô tả. Không lời dẫn, không ngoặc kép bao quanh, không markdown.

## Ví dụ đúng
Ảnh chụp hộp sữa Vinamilk Flex không đường, bao bì ghi thông tin sữa tươi tiệt trùng.
Banner quảng cáo chương trình mua 2 tặng 1 của Nestlé Milo trên nền xanh.
Ảnh cận cảnh chai nước giặt Comfort hương Ban Mai đặt trên nền trắng.
Ảnh chụp màn hình livestream bán mỹ phẩm, người bán giới thiệu kem dưỡng da.
Ảnh chụp một nhóm học sinh trước sân trường trong ngày khai giảng.

## Kết quả OCR của ảnh này (chỉ để tham khảo, có thể sai):
__OCR_CONTEXT__

Nhắc lại: nếu ảnh không có chữ đọc được và không có nhãn hàng/sản phẩm, trả về chuỗi rỗng.
Viết mô tả ngay:'''


VLLM_BASE_URL = 'https://delay-buffer-unnerve.ngrok-free.dev/v1'
VLLM_MODEL = 'Qwen/Qwen3-VL-4B-Instruct'
VLLM_CLIENT = OpenAI(base_url=VLLM_BASE_URL, api_key='EMPTY')
try:
    print(f'vLLM connected: {[m.id for m in VLLM_CLIENT.models.list()]}')
except Exception:
    print(f'vLLM not reachable at {VLLM_BASE_URL}')

[2026-07-31 05:54:23,780] [    INFO] _client.py:1025 - HTTP Request: GET https://delay-buffer-unnerve.ngrok-free.dev/v1/models "HTTP/1.1 200 OK"


vLLM connected: ['Qwen/Qwen3-VL-4B-Instruct']


In [13]:
import random
from threading import Lock

VLM_MAX_RETRIES = 3
VLM_IMG_SIDE = 1024
VLM_TIMEOUT = 120

GATE_LOG, _glock = [], Lock()

PROMPT_GATE = '''Ảnh này có chứa chữ đọc được, hoặc có nhãn hàng, logo, tên sản phẩm nào không? Chỉ trả lời đúng một từ: CÓ hoặc KHÔNG.'''


def _encode(image_pil):
    img = image_pil.copy()
    img.thumbnail((VLM_IMG_SIDE, VLM_IMG_SIDE))
    buf = BytesIO()
    img.save(buf, format='JPEG', quality=90)
    return base64.b64encode(buf.getvalue()).decode()


def _call(b64, prompt, max_tokens):
    last_err = None
    for attempt in range(VLM_MAX_RETRIES):
        try:
            resp = VLLM_CLIENT.chat.completions.create(
                model=VLLM_MODEL,
                max_tokens=max_tokens,
                temperature=0.0,
                timeout=VLM_TIMEOUT,
                extra_body={"repetition_penalty": 1.05},
                messages=[{'role': 'user', 'content': [
                    {'type': 'image_url',
                     'image_url': {'url': 'data:image/jpeg;base64,' + b64}},
                    {'type': 'text', 'text': prompt}]}])
            return resp.choices[0].message.content or ''
        except Exception as e:
            last_err = e
            if attempt < VLM_MAX_RETRIES - 1:
                time.sleep(1.5 * (2 ** attempt) + random.random())
    raise RuntimeError(f'VLM failed after {VLM_MAX_RETRIES} attempts: '
                       f'{type(last_err).__name__}: {last_err}') from last_err


def vlm_summarize(image_pil, ocr_text, box_data=None, image_id=None):
    b64 = _encode(image_pil)

    # Cổng chặn: chỉ chạy khi OCR không đọc được chữ nào (~11% số ảnh).
    # Câu hỏi nhị phân, 4 token, dễ hơn nhiều so với bắt model tự im lặng.
    if not (ocr_text or '').strip():
        raw = _call(b64, PROMPT_GATE, max_tokens=4).strip().upper()
        is_empty = raw.replace('Ô', 'O').lstrip('*# ').startswith('KHONG')
        with _glock:
            GATE_LOG.append({'image_id': image_id, 'gate_raw': raw,
                             'gated_empty': is_empty})
        if is_empty:
            return ''

    context = build_ocr_context(ocr_text, box_data)
    prompt = PROMPT_SUMMARY.replace('__OCR_CONTEXT__', context)
    return _call(b64, prompt, max_tokens=384)


print('vlm_summarize ready (OCR-gated empty check)')

vlm_summarize ready (OCR-gated empty check)


## Test

In [ ]:
import time, numpy as np, collections
from PIL import Image

sample = test_df.sample(n=20, random_state=3)
t_pre = t_det = t_rec = 0.0
counts, longest = [], []

for _, row in sample.iterrows():
    a = time.time()
    img_cv2 = preprocess(Image.open(row['image_path']).convert('RGB'))
    b = time.time()
    r = Detector.predict(img_cv2)
    c = time.time()
    text, boxes = run_ocr(img_cv2)
    d = time.time()

    t_pre += b - a
    t_det += c - b
    t_rec += (d - c) - (c - b)
    counts.append(0 if not r or r[0] is None else len(r[0]['dt_polys']))
    longest.append(max((len(x['text']) for x in boxes), default=0))

n = len(sample)
print(f"preprocess  {t_pre/n*1000:6.0f} ms")
print(f"detection   {t_det/n*1000:6.0f} ms")
print(f"recognition {t_rec/n*1000:6.0f} ms")
print(f"TỔNG        {(t_pre+t_det+t_rec)/n*1000:6.0f} ms/ảnh")

c = np.array(counts)
print(f"\nbox/ảnh — trung vị {int(np.median(c))} | trung bình {c.mean():.1f} "
      f"| p90 {int(np.percentile(c,90))} | max {c.max()}")
print("phân bố:", sorted(collections.Counter(c).items()))
print(f"ký tự dài nhất: {max(longest)}")

preprocess      39 ms
detection       87 ms
recognition    226 ms
TỔNG           352 ms/ảnh

box/ảnh — trung vị 7 | trung bình 16.1 | p90 38 | max 94
phân bố: [(np.int64(2), 2), (np.int64(3), 5), (np.int64(4), 1), (np.int64(6), 1), (np.int64(7), 1), (np.int64(8), 1), (np.int64(11), 1), (np.int64(12), 3), (np.int64(13), 1), (np.int64(15), 1), (np.int64(35), 1), (np.int64(74), 1), (np.int64(94), 1)]
ký tự dài nhất: 62  (nếu = 64 thì REC_MAX_SEQ đang cắt chữ)


## Main Loop - OCR || VLM

In [16]:
N_CONSUMERS = 10
CKPT = 100

total = len(test_df)
print(f'Processing {total} images')

results, rlock = [], Lock()
err_cnt, elock = 0, Lock()
failed, flock = [], Lock()


def producer(tasks, q, n_consumers):
    global err_cnt
    try:
        for iid, ipath in tasks:
            try:
                img_pil = Image.open(ipath).convert('RGB')
                img_cv2 = preprocess(img_pil)
                text, boxes = run_ocr(img_cv2)
                q.put((iid, img_pil, text, boxes))
            except Exception as e:
                with elock:
                    err_cnt += 1
                with flock:
                    failed.append(iid)
                print(f'[P] {iid}: {type(e).__name__}: {e}')
                q.put((iid, None, '', None))
    finally:
        # one sentinel per consumer, even if the producer blew up
        for _ in range(n_consumers):
            q.put(None)


def consumer(q, bar):
    global err_cnt
    while True:
        item = q.get()
        if item is None:
            break
        iid, img_pil, text, boxes = item

        if img_pil is None:
            with rlock:
                results.append({'image_id': iid, 'summary': ''})
            bar.update(1)
            continue

        try:
            summary = vlm_summarize(img_pil, text, boxes, image_id=iid)
        except Exception as e:
            print(f'[C] {iid}: {type(e).__name__}: {e}')
            with elock:
                err_cnt += 1
            summary = ''

        with rlock:
            results.append({'image_id': iid, 'summary': summary})
            cc = len(results)
        bar.update(1)

        if cc % CKPT == 0:
            with rlock:
                snap = list(results)
            t = time.time() - start
            print(f'  CKPT {cc}/{total} | {t/max(1,cc):.2f}s/img | {err_cnt} err')
            with open('submission_checkpoint.jsonl', 'w', encoding='utf-8') as f:
                for r in snap:
                    f.write(json.dumps(r, ensure_ascii=False) + '\n')


tasks = [(row['image_id'], row['image_path']) for _, row in test_df.iterrows()]
print(f'Tasks: {len(tasks)}')

start = time.time()
q = Queue(maxsize=32)
bar = tqdm(total=len(tasks), desc='Processing')

p = Thread(target=producer, args=(tasks, q, N_CONSUMERS), daemon=True)
consumers = [Thread(target=consumer, args=(q, bar), daemon=True)
             for _ in range(N_CONSUMERS)]

p.start()
for c in consumers:
    c.start()
p.join()
for c in consumers:
    c.join()
bar.close()

elapsed = time.time() - start

# guarantee exactly one line per image, even if the producer died early
done = {r['image_id'] for r in results}
missing = [iid for iid in test_df['image_id'] if iid not in done]
for iid in missing:
    results.append({'image_id': iid, 'summary': ''})
if missing:
    print(f'Filled {len(missing)} missing image_ids with empty summary')

# multi-consumer scrambles order; restore dataset order for reproducibility
order = {iid: i for i, iid in enumerate(test_df['image_id'])}
results.sort(key=lambda r: order.get(r['image_id'], 10**9))

print(f'\nDone: {len(results)} images in {elapsed:.1f}s')
print(f'Avg: {elapsed/max(1,len(results)):.2f}s/img  '
      f'TP: {len(results)/max(1,elapsed):.2f} img/s')
if err_cnt:
    print(f'Errors: {err_cnt} - {failed[:10]}')

with open('submission.jsonl', 'w', encoding='utf-8') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'submission.jsonl written ({len(results)} lines)')

Processing 1202 images
Tasks: 1202


Processing:   0%|          | 0/1202 [00:00<?, ?it/s]

[2026-07-31 05:55:24,444] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 05:55:25,899] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 05:55:26,105] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 05:55:26,126] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 05:55:26,167] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 05:55:26,386] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 05:55:26,590] [    INFO] _client.py:1025 - HTTP Requ

  CKPT 100/1202 | 0.30s/img | 0 err


[2026-07-31 05:55:53,495] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 05:55:53,879] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 05:55:54,054] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 05:55:54,176] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 05:55:54,503] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 05:55:54,945] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
/usr/local/lib/python3.12/dist-packages/vietocr/tool/translate.p

...

  CKPT 1100/1202 | 0.45s/img | 0 err


[2026-07-31 06:03:38,255] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 06:03:39,011] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 06:03:39,099] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 06:03:39,467] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 06:03:39,972] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 06:03:40,755] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"
[2026-07-31 06:03:40,994] [    INFO] _client.py:1025 - HTTP Requ

  CKPT 1200/1202 | 0.45s/img | 0 err


[2026-07-31 06:04:26,722] [    INFO] _client.py:1025 - HTTP Request: POST https://delay-buffer-unnerve.ngrok-free.dev/v1/chat/completions "HTTP/1.1 200 OK"



Done: 1202 images in 543.4s
Avg: 0.45s/img  TP: 2.21 img/s
submission.jsonl written (1202 lines)
